# 📊 GUÍA COMPLETA DE ANÁLISIS - TRANSPORTE DE MERCANCÍAS POR CARRETERA EN ESPAÑA

## 🎯 Objetivo
Este notebook proporciona una **guía paso a paso** para extraer la máxima información de los 9 datasets limpios del proyecto.

## 📁 Datasets Disponibles
1. **CO280** - Tráfico total por CCAA, tipo desplazamiento y mercancía
2. **CO282** - Flujos entre comunidades autónomas (origen-destino)
3. **CO285** - Operaciones en vacío (eficiencia del transporte)
4. **CO497** - Índice de precios del transporte
5. **CO516** - Superficie de instalaciones logísticas
6. **CO519** - Transporte de mercancías por modo y ámbito
7. **CO597** - Transporte internacional (toneladas y ton·km)
8. **CO614** - Costes estructurales por tipo de vehículo
9. **IDL** - Índice de desempeño logístico (comparación internacional)

---

## 📦 PASO 1: CONFIGURACIÓN INICIAL Y CARGA DE DATOS

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os

# Configuración de visualización
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configuración de gráficos
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Librerías importadas correctamente")
print(f"📅 Fecha de análisis: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

In [ ]:
# Definir rutas
ruta_processed = '../data/processed'

# Cargar todos los datasets limpios
print("📂 Cargando datasets...\n")

df_co280 = pd.read_csv(os.path.join(ruta_processed, 'CO280_trafico_total_ccaa_tipo_desplaz_y_mercancia_clean.csv'))
print(f"✅ CO280 cargado: {df_co280.shape}")

df_co282 = pd.read_csv(os.path.join(ruta_processed, 'CO282_flujos_ccaa_origen_destino_clean.csv'))
print(f"✅ CO282 cargado: {df_co282.shape}")

df_co285 = pd.read_csv(os.path.join(ruta_processed, 'CO285_operaciones_vacio_clean.csv'))
print(f"✅ CO285 cargado: {df_co285.shape}")

df_co497 = pd.read_csv(os.path.join(ruta_processed, 'CO497_indice_precios_clean.csv'))
print(f"✅ CO497 cargado: {df_co497.shape}")

df_co516 = pd.read_csv(os.path.join(ruta_processed, 'CO516_superficies_logisticas_clean.csv'))
print(f"✅ CO516 cargado: {df_co516.shape}")

df_co519 = pd.read_csv(os.path.join(ruta_processed, 'CO519_transporte_mercancias_por_modo_y_ambito.csv'))
print(f"✅ CO519 cargado: {df_co519.shape}")

df_co597 = pd.read_csv(os.path.join(ruta_processed, 'CO597_transporte_mercancias_internacional.csv'))
print(f"✅ CO597 cargado: {df_co597.shape}")

df_co614 = pd.read_csv(os.path.join(ruta_processed, 'CO614_costes_estructura_clean.csv'))
print(f"✅ CO614 cargado: {df_co614.shape}")

df_lpi = pd.read_csv(os.path.join(ruta_processed, 'indice_desempeno_logistico_clean.csv'))
print(f"✅ IDL cargado: {df_lpi.shape}")

print("\n🎉 ¡Todos los datasets cargados exitosamente!")

---
## 🔍 PASO 2: ANÁLISIS EXPLORATORIO INICIAL (EDA)

### 2.1 Inspección General de Cada Dataset

In [ ]:
# Función para inspección rápida
def inspeccionar_dataset(df, nombre):
    print(f"\n{'='*60}")
    print(f"📊 DATASET: {nombre}")
    print(f"{'='*60}")
    print(f"\n📏 Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"\n📋 Columnas:\n{list(df.columns)}")
    print(f"\n🔢 Tipos de datos:")
    print(df.dtypes)
    print(f"\n❓ Valores nulos:")
    nulos = df.isnull().sum()
    if nulos.sum() > 0:
        print(nulos[nulos > 0])
    else:
        print("✅ No hay valores nulos")
    print(f"\n👀 Primeras 3 filas:")
    display(df.head(3))
    print(f"\n📊 Estadísticas descriptivas (columnas numéricas):")
    display(df.describe())

# Inspeccionar cada dataset
datasets = [
    (df_co280, "CO280 - Tráfico Total por CCAA"),
    (df_co282, "CO282 - Flujos Origen-Destino"),
    (df_co285, "CO285 - Operaciones en Vacío"),
    (df_co497, "CO497 - Índice de Precios"),
    (df_co516, "CO516 - Superficies Logísticas"),
    (df_co519, "CO519 - Transporte por Modo"),
    (df_co597, "CO597 - Transporte Internacional"),
    (df_co614, "CO614 - Costes Estructurales"),
    (df_lpi, "IDL - Índice Desempeño Logístico")
]

for df, nombre in datasets:
    inspeccionar_dataset(df, nombre)

---
## 📈 PASO 3: ANÁLISIS TEMPORAL (SERIES DE TIEMPO)

### 3.1 Evolución del Tráfico Total (2017-2024)

In [ ]:
# CO280: Evolución anual del tráfico total
if 'año' in df_co280.columns and 'toneladas' in df_co280.columns:
    trafico_anual = df_co280.groupby('año')['toneladas'].sum().reset_index()
    trafico_anual['toneladas_millones'] = trafico_anual['toneladas'] / 1_000_000
    
    print("📊 EVOLUCIÓN DEL TRÁFICO TOTAL (2017-2024)\n")
    print(trafico_anual)
    
    # Calcular variación interanual
    trafico_anual['variacion_%'] = trafico_anual['toneladas'].pct_change() * 100
    
    # Visualización
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    # Gráfico 1: Evolución absoluta
    ax1.plot(trafico_anual['año'], trafico_anual['toneladas_millones'], 
             marker='o', linewidth=2, markersize=8, color='#2E86AB')
    ax1.set_title('Evolución del Tráfico Total de Mercancías', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Año', fontsize=12)
    ax1.set_ylabel('Toneladas (millones)', fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    # Añadir valores en los puntos
    for x, y in zip(trafico_anual['año'], trafico_anual['toneladas_millones']):
        ax1.text(x, y, f'{y:.1f}M', ha='center', va='bottom', fontsize=9)
    
    # Gráfico 2: Variación interanual
    colors = ['green' if x > 0 else 'red' for x in trafico_anual['variacion_%'].fillna(0)]
    ax2.bar(trafico_anual['año'], trafico_anual['variacion_%'].fillna(0), color=colors, alpha=0.7)
    ax2.set_title('Variación Interanual del Tráfico (%)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Año', fontsize=12)
    ax2.set_ylabel('Variación (%)', fontsize=12)
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Análisis de tendencia
    print("\n📈 ANÁLISIS DE TENDENCIA:")
    print(f"Tráfico total en 2017: {trafico_anual.iloc[0]['toneladas_millones']:.1f} M toneladas")
    print(f"Tráfico total en 2024: {trafico_anual.iloc[-1]['toneladas_millones']:.1f} M toneladas")
    variacion_total = ((trafico_anual.iloc[-1]['toneladas'] / trafico_anual.iloc[0]['toneladas']) - 1) * 100
    print(f"Variación total 2017-2024: {variacion_total:+.2f}%")
    print(f"Año con mayor tráfico: {trafico_anual.loc[trafico_anual['toneladas'].idxmax(), 'año']}")
    print(f"Año con menor tráfico: {trafico_anual.loc[trafico_anual['toneladas'].idxmin(), 'año']}")

### 3.2 Evolución por Modo de Transporte

In [ ]:
# CO519: Comparación entre modos de transporte
if 'Año' in df_co519.columns and 'Modo Transporte' in df_co519.columns:
    print("🚛 ANÁLISIS POR MODO DE TRANSPORTE\n")
    
    # Crear pivot table
    modos_pivot = df_co519.pivot_table(
        values='Toneladas(miles)', 
        index='Año', 
        columns='Modo Transporte', 
        aggfunc='sum'
    )
    
    # Calcular participación de mercado
    modos_pivot['Total'] = modos_pivot.sum(axis=1)
    for col in modos_pivot.columns[:-1]:
        modos_pivot[f'{col}_pct'] = (modos_pivot[col] / modos_pivot['Total']) * 100
    
    print(modos_pivot)
    
    # Visualización
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Gráfico 1: Evolución absoluta
    modos_pivot.iloc[:, :-1].plot(ax=ax1, marker='o', linewidth=2)
    ax1.set_title('Evolución del Transporte por Modo (Miles de Toneladas)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Año', fontsize=12)
    ax1.set_ylabel('Toneladas (miles)', fontsize=12)
    ax1.legend(title='Modo de Transporte', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Gráfico 2: Participación de mercado
    pct_cols = [col for col in modos_pivot.columns if '_pct' in col]
    modos_pivot[pct_cols].plot(kind='area', stacked=True, ax=ax2, alpha=0.7)
    ax2.set_title('Cuota de Mercado por Modo de Transporte (%)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Año', fontsize=12)
    ax2.set_ylabel('Participación (%)', fontsize=12)
    ax2.set_ylim(0, 100)
    ax2.legend([col.replace('_pct', '') for col in pct_cols], 
               title='Modo de Transporte', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 CUOTA DE MERCADO POR MODO (Promedio 2017-2024):")
    for col in pct_cols:
        modo = col.replace('_pct', '')
        promedio = modos_pivot[col].mean()
        print(f"  {modo}: {promedio:.2f}%")

### 3.3 Evolución de Precios y Costes

In [ ]:
# CO497: Índice de precios del transporte
print("💰 ANÁLISIS DE PRECIOS DEL TRANSPORTE\n")

if 'año' in df_co497.columns:
    print(df_co497)
    
    # Visualización
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Si hay diferentes índices, plotear cada uno
    if 'indice' in df_co497.columns:
        for indice in df_co497['indice'].unique():
            data = df_co497[df_co497['indice'] == indice]
            ax.plot(data['año'], data['valor'], marker='o', linewidth=2, label=indice)
    else:
        ax.plot(df_co497['año'], df_co497['valor'], marker='o', linewidth=2)
    
    ax.set_title('Evolución del Índice de Precios del Transporte', fontsize=14, fontweight='bold')
    ax.set_xlabel('Año', fontsize=12)
    ax.set_ylabel('Índice (Base 100)', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Calcular inflación del transporte
    if 'valor' in df_co497.columns:
        df_co497_sorted = df_co497.sort_values('año')
        variacion_precios = ((df_co497_sorted['valor'].iloc[-1] / df_co497_sorted['valor'].iloc[0]) - 1) * 100
        print(f"\n📈 Variación de precios 2017-2024: {variacion_precios:+.2f}%")

---
## 🗺️ PASO 4: ANÁLISIS GEOGRÁFICO

### 4.1 Tráfico por Comunidad Autónoma

In [ ]:
# CO280: Análisis por CCAA
print("🗺️ ANÁLISIS GEOGRÁFICO POR COMUNIDADES AUTÓNOMAS\n")

if 'ccaa' in df_co280.columns:
    # Tráfico total por CCAA (todos los años)
    trafico_ccaa = df_co280.groupby('ccaa')['toneladas'].sum().sort_values(ascending=False)
    trafico_ccaa_millones = trafico_ccaa / 1_000_000
    
    print("📊 TOP 10 COMUNIDADES AUTÓNOMAS POR TRÁFICO TOTAL:")
    print(trafico_ccaa_millones.head(10))
    
    # Visualización
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    
    # Gráfico 1: Barras horizontales TOP 10
    top10 = trafico_ccaa_millones.head(10)
    colors = plt.cm.viridis(np.linspace(0, 1, len(top10)))
    top10.plot(kind='barh', ax=ax1, color=colors)
    ax1.set_title('TOP 10 CCAA por Tráfico Total (2017-2024)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Toneladas (millones)', fontsize=12)
    ax1.set_ylabel('Comunidad Autónoma', fontsize=12)
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Añadir valores en las barras
    for i, v in enumerate(top10.values):
        ax1.text(v, i, f' {v:.1f}M', va='center', fontsize=10)
    
    # Gráfico 2: Distribución porcentual
    top5 = trafico_ccaa.head(5)
    otros = trafico_ccaa[5:].sum()
    
    data_pie = pd.concat([top5, pd.Series({'Otras': otros})])
    ax2.pie(data_pie.values, labels=data_pie.index, autopct='%1.1f%%', startangle=90)
    ax2.set_title('Distribución del Tráfico (TOP 5 + Otras)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Concentración geográfica
    total_trafico = trafico_ccaa.sum()
    concentracion_top5 = (top5.sum() / total_trafico) * 100
    print(f"\n📍 Concentración geográfica:")
    print(f"  Las TOP 5 CCAA concentran el {concentracion_top5:.1f}% del tráfico total")

### 4.2 Análisis de Flujos Origen-Destino

In [ ]:
# CO282: Matriz origen-destino
print("🔄 ANÁLISIS DE FLUJOS ORIGEN-DESTINO\n")

if 'ccaa_origen' in df_co282.columns and 'ccaa_destino' in df_co282.columns:
    # Crear matriz origen-destino
    if 'toneladas' in df_co282.columns:
        matriz_od = df_co282.pivot_table(
            values='toneladas',
            index='ccaa_origen',
            columns='ccaa_destino',
            aggfunc='sum',
            fill_value=0
        )
        
        # Top 10 rutas más importantes
        flujos_totales = df_co282.groupby(['ccaa_origen', 'ccaa_destino'])['toneladas'].sum()
        top_rutas = flujos_totales.sort_values(ascending=False).head(10)
        
        print("🛣️ TOP 10 RUTAS POR VOLUMEN DE TRÁFICO:\n")
        for (origen, destino), toneladas in top_rutas.items():
            print(f"  {origen:20s} → {destino:20s}: {toneladas/1_000_000:>8.2f} M toneladas")
        
        # Visualización: Heatmap de las principales rutas
        # Seleccionar las 10 CCAA con más tráfico
        ccaa_principales = df_co282.groupby('ccaa_origen')['toneladas'].sum().sort_values(ascending=False).head(10).index
        matriz_top = matriz_od.loc[ccaa_principales, ccaa_principales]
        
        fig, ax = plt.subplots(figsize=(14, 12))
        sns.heatmap(matriz_top/1_000_000, annot=True, fmt='.1f', cmap='YlOrRd', 
                    cbar_kws={'label': 'Toneladas (millones)'}, ax=ax, linewidths=0.5)
        ax.set_title('Matriz Origen-Destino - TOP 10 CCAA\n(Millones de toneladas)', 
                     fontsize=14, fontweight='bold', pad=20)
        ax.set_xlabel('Destino', fontsize=12)
        ax.set_ylabel('Origen', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()
        
        # Análisis de tráfico interno vs externo
        trafico_interno = matriz_od.values.diagonal().sum()
        trafico_total = matriz_od.values.sum()
        trafico_externo = trafico_total - trafico_interno
        
        print(f"\n📊 DISTRIBUCIÓN TRÁFICO INTERNO VS EXTERNO:")
        print(f"  Tráfico interno (dentro de la misma CCAA): {trafico_interno/1_000_000:,.1f} M ton ({(trafico_interno/trafico_total)*100:.1f}%)")
        print(f"  Tráfico externo (entre diferentes CCAA): {trafico_externo/1_000_000:,.1f} M ton ({(trafico_externo/trafico_total)*100:.1f}%)")

### 4.3 Infraestructura Logística por Región

In [ ]:
# CO516: Superficie de instalaciones logísticas
print("🏭 ANÁLISIS DE INFRAESTRUCTURA LOGÍSTICA\n")

if 'CCAA' in df_co516.columns and 'Superficie(m2)' in df_co516.columns:
    # Total por CCAA
    superficie_ccaa = df_co516.groupby('CCAA')['Superficie(m2)'].sum().sort_values(ascending=False)
    superficie_ccaa_millones = superficie_ccaa / 1_000_000
    
    print("📦 SUPERFICIE LOGÍSTICA POR CCAA (millones m²):\n")
    print(superficie_ccaa_millones)
    
    # Visualización por tipo de instalación
    if 'Tipo Instalación' in df_co516.columns:
        tipo_instalacion = df_co516.groupby('Tipo Instalación')['Superficie(m2)'].sum().sort_values(ascending=False)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
        
        # Gráfico 1: Por CCAA
        top10_ccaa = superficie_ccaa_millones.head(10)
        top10_ccaa.plot(kind='bar', ax=ax1, color='steelblue')
        ax1.set_title('TOP 10 CCAA por Superficie Logística', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Comunidad Autónoma', fontsize=12)
        ax1.set_ylabel('Superficie (millones m²)', fontsize=12)
        ax1.tick_params(axis='x', rotation=45)
        ax1.grid(True, alpha=0.3, axis='y')
        
        # Gráfico 2: Por tipo de instalación
        tipo_instalacion_millones = tipo_instalacion / 1_000_000
        tipo_instalacion_millones.plot(kind='bar', ax=ax2, color='coral')
        ax2.set_title('Superficie por Tipo de Instalación', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Tipo de Instalación', fontsize=12)
        ax2.set_ylabel('Superficie (millones m²)', fontsize=12)
        ax2.tick_params(axis='x', rotation=45)
        ax2.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.show()
    
    print(f"\n📍 Total superficie logística en España: {superficie_ccaa.sum()/1_000_000:.1f} millones m²")

---
## 📦 PASO 5: ANÁLISIS DE EFICIENCIA Y OPERACIONES

### 5.1 Operaciones en Vacío (Eficiencia del Transporte)

In [ ]:
# CO285: Análisis de operaciones en vacío
print("🚛 ANÁLISIS DE EFICIENCIA: OPERACIONES EN VACÍO\n")

if 'año' in df_co285.columns:
    print("Columnas disponibles en CO285:", df_co285.columns.tolist())
    print("\nPrimeras filas:")
    display(df_co285.head())
    
    # Calcular ratio de vacío si hay datos de cargado y vacío
    if 'tipo_operacion' in df_co285.columns or 'vacio' in df_co285.columns:
        # Evolución temporal
        evolucion = df_co285.groupby('año').sum(numeric_only=True)
        
        fig, ax = plt.subplots(figsize=(12, 6))
        evolucion.plot(kind='line', marker='o', ax=ax)
        ax.set_title('Evolución de Operaciones en Vacío', fontsize=14, fontweight='bold')
        ax.set_xlabel('Año', fontsize=12)
        ax.set_ylabel('Valor', fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend(title='Tipo')
        plt.tight_layout()
        plt.show()
        
        print("\n⚠️ NOTA: Un alto porcentaje de operaciones en vacío indica ineficiencia en el transporte")

### 5.2 Análisis de Costes Estructurales

In [ ]:
# CO614: Costes estructurales por tipo de vehículo
print("💰 ANÁLISIS DE COSTES ESTRUCTURALES\n")

print("Columnas disponibles:", df_co614.columns.tolist())
print("\nPrimeras filas:")
display(df_co614.head(10))

# Identificar columnas relevantes
if 'año' in df_co614.columns:
    # Evolución de costes por año
    costes_anuales = df_co614.groupby('año').sum(numeric_only=True)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    costes_anuales.plot(kind='line', marker='o', ax=ax)
    ax.set_title('Evolución de Costes Estructurales del Transporte', fontsize=14, fontweight='bold')
    ax.set_xlabel('Año', fontsize=12)
    ax.set_ylabel('Coste (€)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(title='Tipo de Coste', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# Análisis por tipo de vehículo si está disponible
if 'tipo_vehiculo' in df_co614.columns or 'vehiculo' in df_co614.columns:
    col_vehiculo = 'tipo_vehiculo' if 'tipo_vehiculo' in df_co614.columns else 'vehiculo'
    print(f"\n🚚 Tipos de vehículos analizados:")
    print(df_co614[col_vehiculo].value_counts())

---
## 🌍 PASO 6: ANÁLISIS INTERNACIONAL

### 6.1 Transporte Internacional por País

In [ ]:
# CO597: Transporte internacional
print("🌍 ANÁLISIS DEL TRANSPORTE INTERNACIONAL\n")

print("Columnas disponibles:", df_co597.columns.tolist())
display(df_co597.head(10))

if 'pais' in df_co597.columns:
    # Top países por toneladas
    if 'toneladas(miles)' in df_co597.columns:
        top_paises = df_co597.groupby('pais')['toneladas(miles)'].sum().sort_values(ascending=False).head(10)
        
        print("\n🌐 TOP 10 PAÍSES POR VOLUMEN DE INTERCAMBIO:\n")
        for pais, toneladas in top_paises.items():
            print(f"  {pais:20s}: {toneladas:>10,.0f} miles de toneladas")
        
        # Visualización
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
        
        # Gráfico 1: TOP 10 países
        top_paises.plot(kind='barh', ax=ax1, color='teal')
        ax1.set_title('TOP 10 Países por Volumen de Intercambio', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Toneladas (miles)', fontsize=12)
        ax1.set_ylabel('País', fontsize=12)
        ax1.grid(True, alpha=0.3, axis='x')
        
        # Gráfico 2: Recibido vs Expedido (si está disponible)
        if 'desplazamiento' in df_co597.columns:
            flujo_tipo = df_co597.groupby('desplazamiento')['toneladas(miles)'].sum()
            flujo_tipo.plot(kind='pie', ax=ax2, autopct='%1.1f%%', startangle=90)
            ax2.set_title('Distribución: Mercancía Recibida vs Expedida', fontsize=14, fontweight='bold')
            ax2.set_ylabel('')
        
        plt.tight_layout()
        plt.show()
    
    # Análisis de tendencia temporal por país
    if 'año' in df_co597.columns and 'toneladas(miles)' in df_co597.columns:
        print("\n📈 EVOLUCIÓN TEMPORAL TOP 5 PAÍSES:\n")
        
        top5_paises = df_co597.groupby('pais')['toneladas(miles)'].sum().sort_values(ascending=False).head(5).index
        
        fig, ax = plt.subplots(figsize=(14, 7))
        for pais in top5_paises:
            data_pais = df_co597[df_co597['pais'] == pais].groupby('año')['toneladas(miles)'].sum()
            ax.plot(data_pais.index, data_pais.values, marker='o', linewidth=2, label=pais)
        
        ax.set_title('Evolución del Transporte Internacional - TOP 5 Países', fontsize=14, fontweight='bold')
        ax.set_xlabel('Año', fontsize=12)
        ax.set_ylabel('Toneladas (miles)', fontsize=12)
        ax.legend(title='País', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

### 6.2 Comparación con Índice de Desempeño Logístico

In [ ]:
# IDL: Índice de desempeño logístico
print("📊 ÍNDICE DE DESEMPEÑO LOGÍSTICO (LPI)\n")

print("Columnas disponibles:", df_lpi.columns.tolist())
display(df_lpi.head(15))

if 'País' in df_lpi.columns and 'Puntuación' in df_lpi.columns:
    # Comparación de España con otros países
    if 'Indicador' in df_lpi.columns:
        # Filtrar solo el índice global
        lpi_global = df_lpi[df_lpi['Indicador'] == 'Índice global de desempeño logístico (LPI)']
        
        # Crear pivot para comparar países
        lpi_pivot = lpi_global.pivot_table(
            values='Puntuación',
            index='Año',
            columns='País',
            aggfunc='mean'
        )
        
        print("\n🌍 COMPARACIÓN INTERNACIONAL DEL LPI:\n")
        print(lpi_pivot)
        
        # Visualización
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
        
        # Gráfico 1: Evolución temporal
        lpi_pivot.plot(kind='line', marker='o', ax=ax1, linewidth=2)
        ax1.set_title('Evolución del Índice de Desempeño Logístico', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Año', fontsize=12)
        ax1.set_ylabel('Puntuación LPI', fontsize=12)
        ax1.legend(title='País')
        ax1.grid(True, alpha=0.3)
        
        # Gráfico 2: Comparación por componentes (último año)
        ultimo_año = df_lpi['Año'].max()
        lpi_ultimo = df_lpi[df_lpi['Año'] == ultimo_año]
        
        # Crear comparación por indicador
        indicadores_pivot = lpi_ultimo.pivot_table(
            values='Puntuación',
            index='Indicador',
            columns='País',
            aggfunc='mean'
        )
        
        indicadores_pivot.plot(kind='bar', ax=ax2, width=0.8)
        ax2.set_title(f'Comparación por Indicadores ({ultimo_año})', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Indicador', fontsize=12)
        ax2.set_ylabel('Puntuación', fontsize=12)
        ax2.legend(title='País', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax2.tick_params(axis='x', rotation=45)
        ax2.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.show()
        
        # Ranking
        print(f"\n🏆 RANKING LPI {ultimo_año}:")
        ranking = lpi_pivot.loc[ultimo_año].sort_values(ascending=False)
        for i, (pais, puntuacion) in enumerate(ranking.items(), 1):
            print(f"  {i}. {pais}: {puntuacion:.2f}")

---
## 🔗 PASO 7: ANÁLISIS MULTIVARIADO Y CORRELACIONES

### 7.1 Correlación entre Tráfico e Infraestructura

In [ ]:
# Combinar datos de tráfico (CO280) con infraestructura (CO516)
print("🔗 ANÁLISIS DE CORRELACIÓN: TRÁFICO vs INFRAESTRUCTURA\n")

# Preparar datos de tráfico por CCAA
if 'ccaa' in df_co280.columns and 'CCAA' in df_co516.columns:
    trafico_por_ccaa = df_co280.groupby('ccaa')['toneladas'].sum().reset_index()
    trafico_por_ccaa.columns = ['ccaa', 'trafico_total']
    
    # Preparar datos de infraestructura por CCAA
    infra_por_ccaa = df_co516.groupby('CCAA')['Superficie(m2)'].sum().reset_index()
    infra_por_ccaa.columns = ['ccaa', 'superficie_logistica']
    
    # Combinar
    df_correlacion = pd.merge(trafico_por_ccaa, infra_por_ccaa, on='ccaa', how='inner')
    
    # Calcular correlación
    correlacion = df_correlacion['trafico_total'].corr(df_correlacion['superficie_logistica'])
    print(f"📊 Coeficiente de correlación: {correlacion:.3f}")
    
    # Visualización scatter plot
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Convertir a millones para mejor legibilidad
    x = df_correlacion['superficie_logistica'] / 1_000_000
    y = df_correlacion['trafico_total'] / 1_000_000
    
    ax.scatter(x, y, s=100, alpha=0.6, color='darkblue')
    
    # Añadir etiquetas de CCAA
    for i, row in df_correlacion.iterrows():
        ax.annotate(row['ccaa'], 
                   (row['superficie_logistica']/1_000_000, row['trafico_total']/1_000_000),
                   xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    # Añadir línea de tendencia
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)
    ax.plot(x, p(x), "r--", alpha=0.8, linewidth=2, label=f'Tendencia (r={correlacion:.3f})')
    
    ax.set_title('Relación entre Infraestructura Logística y Tráfico de Mercancías por CCAA', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('Superficie Logística (millones m²)', fontsize=12)
    ax.set_ylabel('Tráfico Total (millones toneladas)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Identificar outliers
    df_correlacion['ratio_trafico_superficie'] = df_correlacion['trafico_total'] / df_correlacion['superficie_logistica']
    print("\n📍 CCAA con mayor ratio tráfico/superficie (más intensivo):")
    top_ratio = df_correlacion.nlargest(5, 'ratio_trafico_superficie')[['ccaa', 'ratio_trafico_superficie']]
    for _, row in top_ratio.iterrows():
        print(f"  {row['ccaa']}: {row['ratio_trafico_superficie']:.2f} ton/m²")

### 7.2 Relación entre Precios y Volumen de Tráfico

In [ ]:
# Análisis de relación entre índice de precios y volumen
print("💰 ANÁLISIS: PRECIOS vs VOLUMEN DE TRÁFICO\n")

if 'año' in df_co280.columns and 'año' in df_co497.columns:
    # Volumen anual
    volumen_anual = df_co280.groupby('año')['toneladas'].sum().reset_index()
    volumen_anual.columns = ['año', 'toneladas']
    
    # Índice de precios anual
    if 'valor' in df_co497.columns:
        precios_anual = df_co497.groupby('año')['valor'].mean().reset_index()
        precios_anual.columns = ['año', 'indice_precios']
        
        # Combinar
        df_precio_volumen = pd.merge(volumen_anual, precios_anual, on='año', how='inner')
        
        # Normalizar para comparar tendencias
        df_precio_volumen['toneladas_norm'] = (df_precio_volumen['toneladas'] / df_precio_volumen['toneladas'].iloc[0]) * 100
        df_precio_volumen['precios_norm'] = (df_precio_volumen['indice_precios'] / df_precio_volumen['indice_precios'].iloc[0]) * 100
        
        print(df_precio_volumen)
        
        # Visualización
        fig, ax1 = plt.subplots(figsize=(14, 7))
        
        ax1.set_xlabel('Año', fontsize=12)
        ax1.set_ylabel('Índice de Precios (Base 100)', fontsize=12, color='red')
        line1 = ax1.plot(df_precio_volumen['año'], df_precio_volumen['precios_norm'], 
                        color='red', marker='o', linewidth=2, label='Precios')
        ax1.tick_params(axis='y', labelcolor='red')
        ax1.grid(True, alpha=0.3)
        
        ax2 = ax1.twinx()
        ax2.set_ylabel('Volumen de Tráfico (Base 100)', fontsize=12, color='blue')
        line2 = ax2.plot(df_precio_volumen['año'], df_precio_volumen['toneladas_norm'], 
                        color='blue', marker='s', linewidth=2, label='Volumen')
        ax2.tick_params(axis='y', labelcolor='blue')
        
        # Combinar leyendas
        lines = line1 + line2
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='upper left')
        
        plt.title('Evolución Comparada: Precios vs Volumen de Tráfico (Base 100)', 
                 fontsize=14, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()
        
        # Calcular elasticidad precio-demanda
        variacion_precios = ((df_precio_volumen['indice_precios'].iloc[-1] / df_precio_volumen['indice_precios'].iloc[0]) - 1) * 100
        variacion_volumen = ((df_precio_volumen['toneladas'].iloc[-1] / df_precio_volumen['toneladas'].iloc[0]) - 1) * 100
        
        print(f"\n📊 ANÁLISIS DE ELASTICIDAD:")
        print(f"  Variación de precios 2017-2024: {variacion_precios:+.2f}%")
        print(f"  Variación de volumen 2017-2024: {variacion_volumen:+.2f}%")
        if variacion_precios != 0:
            elasticidad = variacion_volumen / variacion_precios
            print(f"  Elasticidad precio-demanda: {elasticidad:.3f}")
            if abs(elasticidad) > 1:
                print(f"  → Demanda ELÁSTICA (sensible a cambios de precio)")
            else:
                print(f"  → Demanda INELÁSTICA (poco sensible a cambios de precio)")

---
## 📊 PASO 8: ANÁLISIS DE MERCANCÍAS Y TIPOLOGÍA

### 8.1 Distribución por Tipo de Mercancía

In [ ]:
# CO280: Análisis por tipo de mercancía
print("📦 ANÁLISIS POR TIPO DE MERCANCÍA\n")

# Identificar columnas relacionadas con tipo de mercancía
print("Columnas disponibles en CO280:", df_co280.columns.tolist())

# Buscar columna de tipo de mercancía
cols_mercancia = [col for col in df_co280.columns if 'mercancia' in col.lower() or 'tipo' in col.lower()]
if cols_mercancia:
    print(f"\nColumnas de mercancía encontradas: {cols_mercancia}")
    
    for col in cols_mercancia:
        if df_co280[col].nunique() < 50:  # Solo si hay un número razonable de categorías
            print(f"\n📊 Distribución por {col}:")
            dist_mercancia = df_co280.groupby(col)['toneladas'].sum().sort_values(ascending=False)
            dist_mercancia_millones = dist_mercancia / 1_000_000
            
            print(dist_mercancia_millones)
            
            # Visualización TOP 10
            if len(dist_mercancia_millones) > 0:
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
                
                # Gráfico 1: Barras TOP 10
                top10 = dist_mercancia_millones.head(10)
                top10.plot(kind='barh', ax=ax1, color='steelblue')
                ax1.set_title(f'TOP 10 por {col}', fontsize=14, fontweight='bold')
                ax1.set_xlabel('Toneladas (millones)', fontsize=12)
                ax1.grid(True, alpha=0.3, axis='x')
                
                # Gráfico 2: Pie chart TOP 5 + Otros
                top5 = dist_mercancia.head(5)
                otros = dist_mercancia[5:].sum()
                data_pie = pd.concat([top5, pd.Series({'Otros': otros})])
                ax2.pie(data_pie.values, labels=data_pie.index, autopct='%1.1f%%', startangle=90)
                ax2.set_title(f'Distribución: TOP 5 + Otros', fontsize=14, fontweight='bold')
                
                plt.tight_layout()
                plt.show()
                
                # Concentración
                concentracion_top5 = (top5.sum() / dist_mercancia.sum()) * 100
                print(f"\n📍 Las TOP 5 categorías concentran el {concentracion_top5:.1f}% del tráfico")
else:
    print("\n⚠️ No se encontraron columnas específicas de tipo de mercancía en CO280")

### 8.2 Análisis por Tipo de Desplazamiento

In [ ]:
# CO280: Análisis por tipo de desplazamiento
print("🚚 ANÁLISIS POR TIPO DE DESPLAZAMIENTO\n")

# Buscar columna de tipo de desplazamiento
cols_desplazamiento = [col for col in df_co280.columns if 'desplazamiento' in col.lower()]

if cols_desplazamiento:
    col_desp = cols_desplazamiento[0]
    print(f"Analizando: {col_desp}\n")
    
    # Distribución total
    dist_desplazamiento = df_co280.groupby(col_desp)['toneladas'].sum().sort_values(ascending=False)
    dist_desp_millones = dist_desplazamiento / 1_000_000
    
    print("📊 Distribución total por tipo de desplazamiento:")
    print(dist_desp_millones)
    
    # Evolución temporal por tipo
    if 'año' in df_co280.columns:
        evolucion_desp = df_co280.groupby(['año', col_desp])['toneladas'].sum().unstack(fill_value=0)
        evolucion_desp_millones = evolucion_desp / 1_000_000
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
        
        # Gráfico 1: Evolución absoluta
        evolucion_desp_millones.plot(kind='line', marker='o', ax=ax1, linewidth=2)
        ax1.set_title('Evolución por Tipo de Desplazamiento', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Año', fontsize=12)
        ax1.set_ylabel('Toneladas (millones)', fontsize=12)
        ax1.legend(title='Tipo de Desplazamiento')
        ax1.grid(True, alpha=0.3)
        
        # Gráfico 2: Área apilada (proporciones)
        evolucion_desp_millones.plot(kind='area', stacked=True, ax=ax2, alpha=0.7)
        ax2.set_title('Distribución Proporcional por Tipo de Desplazamiento', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Año', fontsize=12)
        ax2.set_ylabel('Toneladas (millones)', fontsize=12)
        ax2.legend(title='Tipo de Desplazamiento', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Análisis de cambios
        print("\n📈 VARIACIÓN 2017-2024 POR TIPO:")
        for tipo in evolucion_desp.columns:
            variacion = ((evolucion_desp[tipo].iloc[-1] / evolucion_desp[tipo].iloc[0]) - 1) * 100
            print(f"  {tipo}: {variacion:+.2f}%")
else:
    print("\n⚠️ No se encontraron columnas de tipo de desplazamiento")

---
## 🎯 PASO 9: ANÁLISIS COMPARATIVO Y BENCHMARKING

### 9.1 Comparación España vs Europa (LPI)

In [ ]:
# Análisis detallado del posicionamiento de España
print("🌍 BENCHMARKING LOGÍSTICO: ESPAÑA vs EUROPA\n")

if 'País' in df_lpi.columns and 'Indicador' in df_lpi.columns:
    # Análisis por indicador para el último año
    ultimo_año = df_lpi['Año'].max()
    df_ultimo = df_lpi[df_lpi['Año'] == ultimo_año]
    
    # Radar chart para comparar países
    # Filtrar solo los subindicadores (excluir el índice global)
    subindicadores = df_ultimo[df_ultimo['Indicador'] != 'Índice global de desempeño logístico (LPI)']
    
    if len(subindicadores) > 0:
        # Preparar datos para radar chart
        from math import pi
        
        paises = subindicadores['País'].unique()
        indicadores = subindicadores['Indicador'].unique()
        
        # Crear figura
        fig = plt.figure(figsize=(14, 14))
        ax = fig.add_subplot(111, projection='polar')
        
        # Ángulos para cada indicador
        angles = [n / float(len(indicadores)) * 2 * pi for n in range(len(indicadores))]
        angles += angles[:1]
        
        # Plot para cada país
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
        for i, pais in enumerate(paises):
            valores = []
            for indicador in indicadores:
                valor = subindicadores[(subindicadores['País'] == pais) & 
                                      (subindicadores['Indicador'] == indicador)]['Puntuación'].values
                valores.append(valor[0] if len(valor) > 0 else 0)
            
            valores += valores[:1]
            ax.plot(angles, valores, 'o-', linewidth=2, label=pais, color=colors[i % len(colors)])
            ax.fill(angles, valores, alpha=0.15, color=colors[i % len(colors)])
        
        # Configurar ejes
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(indicadores, size=10)
        ax.set_ylim(0, 5)
        ax.set_yticks([1, 2, 3, 4, 5])
        ax.set_yticklabels(['1', '2', '3', '4', '5'], size=8)
        ax.grid(True)
        
        plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        plt.title(f'Comparación Multidimensional del Desempeño Logístico ({ultimo_año})', 
                 size=14, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()
    
    # Análisis de fortalezas y debilidades de España
    españa_data = df_ultimo[df_ultimo['País'] == 'España']
    if len(españa_data) > 0:
        print(f"\n🇪🇸 PERFIL DE ESPAÑA ({ultimo_año}):")
        print("\nPuntuaciones por indicador:")
        for _, row in españa_data.iterrows():
            if row['Indicador'] != 'Índice global de desempeño logístico (LPI)':
                print(f"  {row['Indicador']:30s}: {row['Puntuación']:.2f}")
        
        # Identificar fortalezas y debilidades
        subind_españa = españa_data[españa_data['Indicador'] != 'Índice global de desempeño logístico (LPI)']
        mejor_indicador = subind_españa.loc[subind_españa['Puntuación'].idxmax()]
        peor_indicador = subind_españa.loc[subind_españa['Puntuación'].idxmin()]
        
        print(f"\n✅ Fortaleza: {mejor_indicador['Indicador']} ({mejor_indicador['Puntuación']:.2f})")
        print(f"⚠️ Área de mejora: {peor_indicador['Indicador']} ({peor_indicador['Puntuación']:.2f})")

### 9.2 Comparación de Eficiencia entre CCAA

In [ ]:
# Crear índice de eficiencia: tráfico / superficie logística
print("⚡ ÍNDICE DE EFICIENCIA POR CCAA\n")

if 'ccaa' in df_co280.columns and 'CCAA' in df_co516.columns:
    # Tráfico por CCAA
    trafico_ccaa = df_co280.groupby('ccaa')['toneladas'].sum().reset_index()
    trafico_ccaa.columns = ['ccaa', 'trafico']
    
    # Superficie por CCAA
    superficie_ccaa = df_co516.groupby('CCAA')['Superficie(m2)'].sum().reset_index()
    superficie_ccaa.columns = ['ccaa', 'superficie']
    
    # Combinar
    eficiencia = pd.merge(trafico_ccaa, superficie_ccaa, on='ccaa', how='inner')
    
    # Calcular índice de eficiencia (toneladas por m² de superficie logística)
    eficiencia['indice_eficiencia'] = eficiencia['trafico'] / eficiencia['superficie']
    eficiencia = eficiencia.sort_values('indice_eficiencia', ascending=False)
    
    print("📊 Índice de eficiencia por CCAA (toneladas / m² superficie):")
    print(eficiencia[['ccaa', 'indice_eficiencia']].round(2))
    
    # Visualización
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = ['green' if x > eficiencia['indice_eficiencia'].median() else 'orange' 
             for x in eficiencia['indice_eficiencia']]
    
    bars = ax.barh(range(len(eficiencia)), eficiencia['indice_eficiencia'], color=colors, alpha=0.7)
    ax.set_yticks(range(len(eficiencia)))
    ax.set_yticklabels(eficiencia['ccaa'])
    ax.set_xlabel('Índice de Eficiencia (ton/m²)', fontsize=12)
    ax.set_title('Índice de Eficiencia Logística por CCAA\n(Tráfico / Superficie Logística)', 
                fontsize=14, fontweight='bold')
    ax.axvline(eficiencia['indice_eficiencia'].median(), color='red', 
              linestyle='--', linewidth=2, label='Mediana')
    ax.grid(True, alpha=0.3, axis='x')
    ax.legend()
    
    # Añadir valores
    for i, v in enumerate(eficiencia['indice_eficiencia']):
        ax.text(v, i, f' {v:.2f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📈 Estadísticas del índice de eficiencia:")
    print(f"  Media: {eficiencia['indice_eficiencia'].mean():.2f}")
    print(f"  Mediana: {eficiencia['indice_eficiencia'].median():.2f}")
    print(f"  Desviación estándar: {eficiencia['indice_eficiencia'].std():.2f}")

---
## 📝 PASO 10: RESUMEN EJECUTIVO Y CONCLUSIONES

### 10.1 Generación de Resumen Automático

In [ ]:
# Generar resumen ejecutivo con los principales hallazgos
print("="*80)
print(" "*20 + "📊 RESUMEN EJECUTIVO DEL ANÁLISIS")
print("="*80)
print()

# 1. Período analizado
if 'año' in df_co280.columns:
    años = sorted(df_co280['año'].unique())
    print(f"📅 PERÍODO ANALIZADO: {años[0]} - {años[-1]}")
    print()

# 2. Volumen total de tráfico
trafico_total = df_co280['toneladas'].sum() / 1_000_000
print(f"🚛 VOLUMEN TOTAL DE TRÁFICO: {trafico_total:,.0f} millones de toneladas")
print()

# 3. Top CCAA
if 'ccaa' in df_co280.columns:
    top3_ccaa = df_co280.groupby('ccaa')['toneladas'].sum().sort_values(ascending=False).head(3)
    print("🗺️ TOP 3 COMUNIDADES AUTÓNOMAS:")
    for i, (ccaa, toneladas) in enumerate(top3_ccaa.items(), 1):
        pct = (toneladas / df_co280['toneladas'].sum()) * 100
        print(f"   {i}. {ccaa}: {toneladas/1_000_000:.1f}M ton ({pct:.1f}%)")
    print()

# 4. Infraestructura
if 'Superficie(m2)' in df_co516.columns:
    superficie_total = df_co516['Superficie(m2)'].sum() / 1_000_000
    print(f"🏭 SUPERFICIE LOGÍSTICA TOTAL: {superficie_total:.1f} millones m²")
    print()

# 5. Transporte internacional
if 'toneladas(miles)' in df_co597.columns:
    top_pais = df_co597.groupby('pais')['toneladas(miles)'].sum().sort_values(ascending=False).head(1)
    print(f"🌍 PRINCIPAL SOCIO COMERCIAL: {top_pais.index[0]}")
    print()

# 6. Posicionamiento LPI
if 'País' in df_lpi.columns and 'Puntuación' in df_lpi.columns:
    ultimo_año_lpi = df_lpi['Año'].max()
    españa_lpi = df_lpi[(df_lpi['País'] == 'España') & 
                        (df_lpi['Año'] == ultimo_año_lpi) & 
                        (df_lpi['Indicador'] == 'Índice global de desempeño logístico (LPI)')]
    if len(españa_lpi) > 0:
        puntuacion = españa_lpi['Puntuación'].values[0]
        print(f"📊 ÍNDICE LPI ESPAÑA ({ultimo_año_lpi}): {puntuacion:.2f}/5.00")
        print()

# 7. Tendencia general
if 'año' in df_co280.columns:
    trafico_inicial = df_co280[df_co280['año'] == años[0]]['toneladas'].sum()
    trafico_final = df_co280[df_co280['año'] == años[-1]]['toneladas'].sum()
    variacion_total = ((trafico_final / trafico_inicial) - 1) * 100
    print(f"📈 VARIACIÓN TOTAL DEL TRÁFICO ({años[0]}-{años[-1]}): {variacion_total:+.2f}%")
    print()

print("="*80)
print(f"\n✅ Análisis completado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n💡 Este análisis ha extraído información de 9 datasets diferentes,")
print("   proporcionando una visión 360° del transporte de mercancías en España.")

### 10.2 Exportar Resultados Clave

In [ ]:
# Crear un DataFrame consolidado con los KPIs principales
print("💾 EXPORTANDO RESULTADOS CLAVE...\n")

# Crear diccionario de KPIs
kpis = {}

# Tráfico anual
if 'año' in df_co280.columns:
    trafico_anual = df_co280.groupby('año')['toneladas'].sum()
    for año, valor in trafico_anual.items():
        kpis[f'trafico_total_{año}'] = valor

# Tráfico por CCAA
if 'ccaa' in df_co280.columns:
    trafico_ccaa = df_co280.groupby('ccaa')['toneladas'].sum()
    for ccaa, valor in trafico_ccaa.items():
        kpis[f'trafico_{ccaa.replace(" ", "_")}'] = valor

# Superficie logística por CCAA
if 'CCAA' in df_co516.columns:
    superficie_ccaa = df_co516.groupby('CCAA')['Superficie(m2)'].sum()
    for ccaa, valor in superficie_ccaa.items():
        kpis[f'superficie_{ccaa.replace(" ", "_")}'] = valor

# Convertir a DataFrame
df_kpis = pd.DataFrame(list(kpis.items()), columns=['Indicador', 'Valor'])

# Exportar
ruta_salida = '../reports/kpis_principales.csv'
df_kpis.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
print(f"✅ KPIs exportados a: {ruta_salida}")

print(f"\n📊 Total de indicadores exportados: {len(df_kpis)}")
print("\n✨ ¡Análisis completo finalizado!")

---
## 🎓 CONCLUSIONES Y PRÓXIMOS PASOS

### Información Extraída:

1. **Análisis Temporal**: Evolución del tráfico, precios y costes (2017-2024)
2. **Análisis Geográfico**: Distribución por CCAA, flujos origen-destino, concentración
3. **Análisis Modal**: Comparación entre carretera, ferrocarril, marítimo y aéreo
4. **Análisis de Eficiencia**: Operaciones en vacío, ratio tráfico/infraestructura
5. **Análisis Internacional**: Principales socios comerciales, flujos de importación/exportación
6. **Benchmarking**: Comparación con países europeos mediante LPI
7. **Análisis de Mercancías**: Distribución por tipo de producto y desplazamiento
8. **Correlaciones**: Relación entre infraestructura, tráfico, precios y costes

### Próximos Pasos Recomendados:

1. **Modelado Predictivo**: Series temporales (ARIMA, Prophet) para proyecciones futuras
2. **Análisis de Redes**: Visualización de flujos con NetworkX
3. **Clustering**: Segmentación de CCAA por características logísticas
4. **Dashboard Interactivo**: Implementación en Tableau/Power BI/Streamlit
5. **Análisis de Impacto COVID-19**: Estudio específico del período 2020-2021
6. **Optimización de Rutas**: Análisis de eficiencia en las principales rutas comerciales

---

**📌 NOTA**: Este notebook proporciona una guía completa para el análisis. 
Puedes ejecutar las celdas secuencialmente o seleccionar solo las secciones de tu interés.
